# 📈 Hệ Thống Paper Trading (Giao Dịch Ảo)

Notebook này được thiết kế để nạp lại bộ não AI (mô hình RL PPO) mới nhất từ file `.zip` và mô phỏng giao dịch thực chiến (Paper Trading). Bạn có thể kiểm thử danh mục đầu tư với vốn ảo trước khi áp dụng vào tài khoản thật.

In [ ]:

import os
import sys
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

# Thêm thư mục gốc vào sys.path để có thể import từ src và model
sys.path.append(os.path.abspath('..'))

# Chú ý: Đã đổi sang import từ file v7_3 để tương thích với model v7_3
from model.ppo_grouped_rl_v7_3 import CONFIG, AdvancedTickerExtractor, load_data, AdvancedPortfolioEnv

print("Khoi tao moi truong thanh cong!")

🎲 Random Seed: 321
Khoi tao moi truong thanh cong!


## 1. Nạp Mô Hình AI Mới Nhất & Dữ Liệu

In [10]:
# Nạp dữ liệu thị trường mới nhất
print("Dang lay du lieu thi truong moi nhat...")
returns_df, ai_features_df, strategies_features_df, weights_dim, tickers, num_strategies_features, dates = load_data()

# Khởi tạo môi trường ảo để format input cho AI
def make_env():
    return AdvancedPortfolioEnv(
        returns_df,
        ai_features_df,
        strategies_features_df,
        weights_dim,
        num_strategies_features,
        tickers=tickers,
        dates=dates,
        is_test=True
    )

env = DummyVecEnv([make_env])

# Nạp VecNormalize (Cực kỳ quan trọng để chuẩn hóa input)
NORMALIZE_PATH = '../model/v7_3/vec_normalize.pkl' # Dùng bản v7_3
if os.path.exists(NORMALIZE_PATH):
    env = VecNormalize.load(NORMALIZE_PATH, env)
    env.training = False
    env.norm_reward = False
    print("Da nap thanh cong bo chuan hoa (VecNormalize)!")

# Nạp mô hình AI
MODEL_PATH = '../model/v7_3/AI_Brain_v7_3.zip' # Dùng bản v7_3
print(f"Dang nap bo nao AI tu: {MODEL_PATH} ...")
model = PPO.load(MODEL_PATH)
print("Da nap thanh cong bo nao AI!")

Dang lay du lieu thi truong moi nhat...
Da nap thanh cong bo chuan hoa (VecNormalize)!
Dang nap bo nao AI tu: ../model/v7_3/AI_Brain_v7_3.zip ...
Da nap thanh cong bo nao AI!


## 2. Dự Báo Danh Mục & Paper Trading
Chạy mô hình AI trên dữ liệu hôm nay để lấy danh mục (Portfolio Weights) cho ngày mai.

In [11]:
VON_KHOI_DIEM = 100_000_000  # 100 triệu VND vốn ảo

def paper_trading_predict(env, tickers, capital):
    """Dự báo hành động và in ra lệnh giao dịch cụ thể"""
    obs = env.reset()
    print("AI dang tinh toan ty trong danh muc toi uu...")
    
    # AI Dự báo tỷ trọng
    action, _states = model.predict(obs, deterministic=True)
    action = action[0] # Lấy action từ DummyVecEnv
    
    print(f"\n{'='*40}")
    print(f"DE XUAT GIAO DICH CHO NGAY T+1")
    print(f"{'='*40}")
    print(f"Von hien tai: {capital:,.0f} VND\n")
    
    for idx, ticker in enumerate(tickers):
        weight = action[idx]
        if weight > 0.01: # Bỏ qua các mã có tỷ trọng < 1%
            allocated_cash = capital * weight
            print(f"> MUA/NAM GIU {ticker}: {weight*100:5.2f}% danh muc (tuong duong {allocated_cash:,.0f} VND)")
            
    print(f"\nGhi chu: Su dung cac ty le nay de dat lenh tren he thong Paper Trading.")

# Chạy dự báo
paper_trading_predict(env=env, tickers=tickers, capital=VON_KHOI_DIEM)


AI dang tinh toan ty trong danh muc toi uu...

DE XUAT GIAO DICH CHO NGAY T+1
Von hien tai: 100,000,000 VND

> MUA/NAM GIU CII:  5.77% danh muc (tuong duong 5,768,892 VND)
> MUA/NAM GIU DCM:  6.64% danh muc (tuong duong 6,637,092 VND)
> MUA/NAM GIU EIB: 11.55% danh muc (tuong duong 11,550,470 VND)
> MUA/NAM GIU GMD: 12.73% danh muc (tuong duong 12,725,204 VND)
> MUA/NAM GIU HAG: 14.79% danh muc (tuong duong 14,794,655 VND)
> MUA/NAM GIU KBC: 11.96% danh muc (tuong duong 11,957,686 VND)
> MUA/NAM GIU KDH: 22.81% danh muc (tuong duong 22,813,606 VND)
> MUA/NAM GIU MBB:  5.47% danh muc (tuong duong 5,474,290 VND)
> MUA/NAM GIU NLG:  5.99% danh muc (tuong duong 5,985,940 VND)
> MUA/NAM GIU PDR: 10.70% danh muc (tuong duong 10,699,171 VND)
> MUA/NAM GIU PHR:  9.67% danh muc (tuong duong 9,672,435 VND)
> MUA/NAM GIU PNJ: 21.54% danh muc (tuong duong 21,538,966 VND)
> MUA/NAM GIU PVD:  5.35% danh muc (tuong duong 5,345,407 VND)
> MUA/NAM GIU SSI: 11.79% danh muc (tuong duong 11,785,795 VND)

